# 07. Your Own Classifier — Feature Extraction + Training with numpy

> Training a big model from scratch needs hundreds of thousands of photos and days of GPU time. Instead, we borrow **the eyes of a model that has already learned** (transfer learning).
> Those eyes sum up each photo as 1280 numbers. On top of that, we train only a **very small model** with a few dozen of our own photos.
> This is the same thing the [Teach] screen in edge-lab does (there, training happens in the browser).

### What you will learn
1. **Feature extraction** — using MobileNetV2 up to just before its last decision layer
2. **Softmax regression** — building it yourself, from the formulas to numpy code (gradient descent)
3. Train / test split, accuracy, **confusion matrix**, overfitting
4. Comparing with a simpler method (nearest mean)

### The whole flow
```
photo ─▶ [MobileNetV2 (fixed, NPU)] ─▶ 1280 numbers ─▶ [softmax regression W(1280×K), b(K)] ─▶ K probabilities
          already-trained eyes (fixed)   features         the part we train (numpy)
```

## 1. The feature model
`mobilenetv2_feat` is MobileNetV2 with its last layer (the one that decides "which of 1000 things is this?") removed.
So its output is not an answer but **1280 numbers that sum up the photo**. Similar photos give similar numbers.
The input is `(1, 224, 224, 3)` — **NHWC**, RGB, 0~1 (the same preprocessing as TF.js on the Teach screen).

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

def pick(*prefer):
    """Pick the first device in the list that this PC has."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

PHOTO = "sample.jpg"          # photo to use when there is no webcam (same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV photo (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Take one photo from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark images for the first few frames
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} — put a photo file name in PHOTO"
        print("No webcam, so using the file:", PHOTO)
    return frame


DEVICE = pick("NPU", "GPU", "CPU")
backbone = core.compile_model(MODELS / "backbone" / "mobilenetv2_feat.xml", DEVICE)
print(DEVICE, "input:", backbone.input(0).shape, "output:", backbone.output(0).shape)

def feature(bgr):
    """Photo → 1280 numbers."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = cv2.resize(rgb, (224, 224), interpolation=cv2.INTER_NEAREST).astype(np.float32) / 255.0
    return backbone(x[None])[backbone.output(0)].reshape(-1)

## 2. Collecting photos
Write your classes in `LABELS`. For each class, the webcam takes `N` photos with a short gap between them — **while it does, change the object's angle · distance · position**.
The more variety you capture, the better it works on new photos. If there is no webcam, it reads from the `data/<class>/*.jpg` folders.

`input()` shows an input box inside the notebook — press Enter to start taking photos.

In [ ]:
LABELS = ["scissors", "rock", "paper"]
N = 20

def capture(n, index=0, gap=0.15):
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    shots = []
    for _ in range(5):
        cap.read()                                    # throw away the first few frames
    while len(shots) < n:
        ok, f = cap.read()
        if not ok:
            break
        shots.append(f)
        time.sleep(gap)
    cap.release()
    return shots

X, y = [], []
for k, label in enumerate(LABELS):
    folder = Path("data") / label
    if folder.is_dir():
        shots = [cv2.imread(str(p)) for p in sorted(folder.glob("*.jpg"))]
    else:
        input(f"Show '{label}' to the webcam and press Enter")
        shots = capture(N)
    shots = [s for s in shots if s is not None]
    print(f"{label}: {len(shots)} photos")
    for s in shots:
        X.append(feature(s))
        y.append(k)
X, y = np.array(X, np.float32), np.array(y)
print("Feature matrix X:", X.shape, "| answers y:", y.shape)

### Seeing the features — squashing 1280 dimensions down to 2 (PCA)
We can't draw 1280 numbers, so we squash them onto the two directions where they spread out the most, and plot them as dots.
If the dots of each class **cluster separately**, the classes are easy to tell apart.

In [ ]:
import matplotlib.pyplot as plt     # graph text is in English (Korean would need extra font settings)
Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)    # PCA = SVD of the centered matrix
P = Xc @ Vt[:2].T                                    # project onto two directions
plt.figure(figsize=(4.5, 4))
for k in range(len(LABELS)):
    plt.scatter(P[y == k, 0], P[y == k, 1], s=18, label=f"class {k}")
plt.legend(); plt.title("features (PCA 2D)"); plt.show()
print("Class numbers:", {k: l for k, l in enumerate(LABELS)})

## 3. Softmax regression — from formulas to code

**Scores**: one score per class. $s = x W + b$ ($x$: 1280 features, $W$: 1280 × K, $b$: K numbers)

**Probabilities** (softmax): turn the scores into values between 0~1 that add up to 1. $p_k = \dfrac{e^{s_k}}{\sum_j e^{s_j}}$

**Loss** (cross entropy): the closer the correct class's probability is to 1, the smaller the loss. $L = -\log p_{\text{correct}}$

**Gradient**: surprisingly simple. $\dfrac{\partial L}{\partial s} = p - \text{onehot}(\text{correct})$ → $\dfrac{\partial L}{\partial W} = x^\top (p - \text{onehot})$

**Gradient descent**: $W \leftarrow W - \eta \cdot \text{gradient}$ ($\eta$: learning rate `lr`). We fix things little by little in the direction that lowers the loss.

**Regularization** (`lam`): shrink $W$ a little so it doesn't grow too large — this reduces **overfitting** (fitting only the training photos).

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
cut = int(len(X) * 0.8)                               # learn with 80%, test with 20%
tr, te = idx[:cut], idx[cut:]

mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6           # put every feature on the same scale (use only the training photos' mean · std!)
Z = (X - mu) / sd
K, D = len(LABELS), X.shape[1]
W = np.zeros((D, K), np.float32)
b = np.zeros(K, np.float32)

def softmax(s):
    s = s - s.max(1, keepdims=True)                   # avoid overflow from big numbers (same result)
    e = np.exp(s)
    return e / e.sum(1, keepdims=True)

onehot = np.eye(K)[y]
lr, lam = 0.05, 1e-3
history = []
for step in range(301):
    p = softmax(Z[tr] @ W + b)
    loss = float(-np.log(np.clip(p[np.arange(len(tr)), y[tr]], 1e-9, 1)).mean()) + 0.0   # + 0.0: turns -0.0 into 0.0
    g = (p - onehot[tr]) / len(tr)                    # gradient with respect to the scores
    W -= lr * (Z[tr].T @ g + lam * W)
    b -= lr * g.sum(0)
    history.append(loss)
    if step % 50 == 0:
        acc_tr = (p.argmax(1) == y[tr]).mean()
        acc_te = (softmax(Z[te] @ W + b).argmax(1) == y[te]).mean() if len(te) else float("nan")
        print(f"{step:3d}  loss {loss:.3f}  train {acc_tr:.0%}  test {acc_te:.0%}")

plt.figure(figsize=(4.5, 2.4)); plt.plot(history); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

**How to read it** — "train" is the accuracy on the photos used for training. "test" is the accuracy on photos the model **has never seen**. The test score shows the real skill.
If train is 100% but test is low, that is **overfitting**: collect more varied photos or make `lam` bigger.

## 4. Confusion matrix — what gets mixed up with what
Rows = correct answer, columns = the model's answer. The diagonal is what it got right; everything else is what it mixed up.

In [ ]:
pred_te = softmax(Z[te] @ W + b).argmax(1)
cm = np.zeros((K, K), int)
for t_, p_ in zip(y[te], pred_te):
    cm[t_, p_] += 1
print("true \\ pred ", "  ".join(f"{l:>4s}" for l in LABELS))
for k in range(K):
    print(f"{LABELS[k]:>8s}  ", "  ".join(f"{v:4d}" for v in cm[k]))

## 5. A simpler method — nearest mean
Compute the **mean** feature of each class. A new photo gets the class of the nearest mean. There is hardly any training — just a few lines.
When you have few photos, this also works pretty well. Let's compare.

In [ ]:
centroids = np.stack([Z[tr][y[tr] == k].mean(0) for k in range(K)])          # mean of each class (K × 1280)
d = ((Z[te][:, None, :] - centroids[None]) ** 2).sum(-1)                       # distance: test photos × classes
acc_nc = (d.argmin(1) == y[te]).mean() if len(te) else float("nan")
acc_sm = (pred_te == y[te]).mean() if len(te) else float("nan")
print(f"Test accuracy — softmax regression {acc_sm:.0%} · nearest mean {acc_nc:.0%}")

## 6. Try it on a new photo

In [ ]:
def predict(bgr):
    p = softmax(((feature(bgr) - mu) / sd)[None] @ W + b)[0]
    return LABELS[int(p.argmax())], p

shot = grab()
label, p = predict(shot)
show(shot, 320)
print("Answer:", label, "| probabilities:", {l: round(float(v), 2) for l, v in zip(LABELS, p)})

## 7. Save · load
All we trained is `W`, `b`, plus `mu`, `sd` used for normalization — so that is all we need to save (we keep using the feature model as it is).

In [ ]:
np.savez("my_classifier.npz", W=W, b=b, mu=mu, sd=sd, labels=np.array(LABELS))
m = np.load("my_classifier.npz")
print("Saved classes:", list(m["labels"]), "| W", m["W"].shape, "| file size", round(Path("my_classifier.npz").stat().st_size / 1024), "KB")

## Common errors
| Symptom | Cause · fix |
|---|---|
| Loss is `nan` | `lr` is too big → try 0.01 |
| Only test accuracy is low | Overfitting · photos are too similar → take more varied photos, make `lam` bigger |
| Always the same answer on new photos | The background was different for each class → the model learned the background instead of the object. Take all photos on the same background |
| Test accuracy is `nan` | Too few photos, so the test share is 0 photos |

## Summary
- Transfer learning: the big model is **fixed eyes** (feature extraction). We train only a **small decision layer** on top.
- Softmax regression = scores (xW+b) → probabilities (softmax) → loss (cross entropy) → gradient (p − correct) → gradient descent.
- Measure skill on photos the model has never seen (test). Use the confusion matrix to see what it mixes up.

## Try it
1. What happens to test accuracy if you lower `N` to 5? What if you raise it to 40?
2. How does the loss graph change if you set `lr` to 0.5 · 0.005?
3. Add one more class (for example, "nothing"). Why would this be useful?
   <details><summary>Hint</summary>

   With only three classes, the model picks one of the three even when you show it nothing. With a "nothing" class, it can choose that.
   </details>
4. Swap the features for the 63 hand-dot numbers from notebook 06 and build a hand-gesture classifier. Does it still work when the background changes?

---
# 🛠 Mini project — Real-time judge — with "not sure"

**What to build**: **load** the saved classifier (`my_classifier.npz`) **again** and judge the webcam video in real time.
If the highest probability is below a threshold (`MIN_P`), it does not force a choice — it says **"?"**. It records every time the result changes.

**What you use**: `feature` · `softmax` (from this notebook), `np.load`, `live`

**Why we need "not sure"** — softmax always pushes the probability onto one of the three classes. Even for an object it has never seen, it answers something like "scissors 0.5".
Just throwing away low-probability answers removes a lot of wrong answers.

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """For `seconds` seconds, turn webcam frames into images with process(photo) → photo and show them in the notebook. To stop, press I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # no webcam → use the photo
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n} frames · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (no webcam, so faking it with the photo)" if still is not None else ""))
    return out

In [ ]:
m = np.load("my_classifier.npz")                      # load what we saved (no new training)
W_, b_, mu_, sd_, labels_ = m["W"], m["b"], m["mu"], m["sd"], [str(l) for l in m["labels"]]
MIN_P = 0.6
changes, prev = [], [None]

def judge(f):
    p = softmax(((feature(f) - mu_) / sd_)[None] @ W_ + b_)[0]
    k = int(p.argmax())
    tag = f"class {k}" if p[k] >= MIN_P else "?"       # text on the image stays short ASCII (cv2 can't draw Korean)
    if tag != prev[0]:
        changes.append((time.strftime("%H:%M:%S"), labels_[k] if p[k] >= MIN_P else "not sure", round(float(p[k]), 2)))
        prev[0] = tag
    color = (0, 200, 0) if tag != "?" else (0, 0, 255)
    cv2.putText(f, f"{tag}  {p[k]:.2f}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)
    for i, v in enumerate(p):                          # probability bars
        cv2.rectangle(f, (10, 60 + i * 22), (10 + int(v * 200), 78 + i * 22), (255, 160, 0), -1)
    return f

live(judge, seconds=15)
print("Class numbers:", {f"class {i}": l for i, l in enumerate(labels_)})
print("When the result changed:", changes)

**Try more**
- Count a result as "confirmed" only when it stays the same for 1 second or more (stops flickering)
- When a result is confirmed, send a command to the board, like in notebook 11
- Change `MIN_P` to 0.4 · 0.8 and count how often "?" shows up